In [ ]:
# importing dependancies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
!pip install lightgbm xgboost
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [ ]:
# loading datasets
training_dataset = pd.read_csv('https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/train.csv')
testing_dataset = pd.read_csv('https://raw.githubusercontent.com/atharv-d21/traffic_demand_prediction/refs/heads/main/datasets/test.csv')
training_dataset.head(5)

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


# DATA ANALYSIS

In [ ]:
training_dataset['day'].unique()

array([48, 49])

In [ ]:
training_dataset['RoadType'].unique()

array([nan, 'Residential', 'Street', 'Highway'], dtype=object)

In [ ]:
training_dataset['NumberofLanes'].unique()

array([1, 3, 2, 4, 5])

In [ ]:
training_dataset['LargeVehicles'].unique()

array(['Not Allowed', 'Allowed'], dtype=object)

In [ ]:
training_dataset['Landmarks'].unique()

array(['No', 'Yes'], dtype=object)

In [ ]:
training_dataset['Temperature'].unique()

array([        nan, 31.1045646 , 25.91926732, ..., 19.67885975,
       22.57395805,  1.32203433])

In [ ]:
training_dataset['Temperature'].max()

48.2514326535093

In [ ]:
training_dataset['Weather'].unique()

array([nan, 'Sunny', 'Rainy', 'Foggy', 'Snowy'], dtype=object)

In [ ]:
training_dataset['demand'].min(), training_dataset['demand'].max()

(6.245650130093708e-07, 1.0)

In [ ]:
training_dataset['Index'][training_dataset['Temperature']==training_dataset['Temperature'].min()]

,Index
8636,8636


In [ ]:
training_dataset['RoadType'].value_counts(dropna=False)

,count
RoadType,
Residential,69230
Street,3909
Highway,3560
NaN,600


In [ ]:
training_dataset['Weather'].value_counts(dropna=False)

,count
Weather,
Sunny,27717
Rainy,20824
Foggy,20243
Snowy,7718
NaN,797


# FEATURE ENGINEERING

In [ ]:
# spliting the 'geohast' into multiple features
training_dataset['geohash_prefix'] = training_dataset['geohash'].str[:3]
training_dataset['geohash_location'] = training_dataset['geohash'].str[3:]

testing_dataset['geohash_prefix'] = testing_dataset['geohash'].str[:3]
testing_dataset['geohash_location'] = testing_dataset['geohash'].str[3:]

display(training_dataset.head(5))

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_prefix,geohash_location
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,qp0,2z1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,qp0,2zt
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,qp0,8bj
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,qp0,8gt
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,qp0,2zq


In [ ]:
training_dataset['geohash_location'].value_counts()

,count
geohash_location,
3wd,105
3wf,105
9t0,105
3w9,105
3x3,105
...,...
8gs,1
8fq,1
d1t,1


In [ ]:
# dropping 'geohash', 'geohash_prefix'
training_dataset.drop(['geohash', 'geohash_prefix'], axis=1, inplace=True)
testing_dataset.drop(['geohash', 'geohash_prefix'], axis=1, inplace=True)

training_dataset.head(5)

,Index,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location
0,0,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,2z1
1,1,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,2zt
2,2,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,8bj
3,3,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,8gt
4,4,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,2zq


In [ ]:
training_dataset[['hour', 'minute']] = training_dataset['timestamp'].str.split(':', expand=True)
testing_dataset[['hour', 'minute']] = testing_dataset['timestamp'].str.split(':', expand=True)

# Convert 'hour' and 'minute' to numeric types
training_dataset['hour'] = pd.to_numeric(training_dataset['hour'])
training_dataset['minute'] = pd.to_numeric(training_dataset['minute'])
testing_dataset['hour'] = pd.to_numeric(testing_dataset['hour'])
testing_dataset['minute'] = pd.to_numeric(testing_dataset['minute'])

display(training_dataset.head(5))

,Index,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location,hour,minute
0,0,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,2z1,0,0
1,1,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,2zt,0,0
2,2,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,8bj,0,0
3,3,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,8gt,0,0
4,4,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,2zq,0,0


In [ ]:
# dropping 'timestamp'
training_dataset.drop(['timestamp'], axis=1, inplace=True)
testing_dataset.drop(['timestamp'], axis=1, inplace=True)

training_dataset.head(5)

,Index,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location,hour,minute
0,0,48,0.048804,NaN,1,Not Allowed,No,NaN,NaN,2z1,0,0
1,1,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,2zt,0,0
2,2,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,8bj,0,0
3,3,48,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,8gt,0,0
4,4,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,2zq,0,0


In [ ]:
# --- Part of Day Feature ---
def get_part_of_day(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

training_dataset['part_of_day'] = training_dataset['hour'].apply(get_part_of_day)
testing_dataset['part_of_day'] = testing_dataset['hour'].apply(get_part_of_day)

display(training_dataset.head())

,Index,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location,hour,minute,part_of_day
0,0,48,0.048804,NaN,1,Not Allowed,No,NaN,NaN,2z1,0,0,Night
1,1,48,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,2zt,0,0,Night
2,2,48,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,8bj,0,0,Night
3,3,48,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,8gt,0,0,Night
4,4,48,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,2zq,0,0,Night


In [ ]:
# fill null values with 'Unknown'
training_dataset['RoadType'] = training_dataset['RoadType'].fillna("Unknown")
testing_dataset['RoadType'] = testing_dataset['RoadType'].fillna("Unknown")

training_dataset['Weather'] = training_dataset['Weather'].fillna("Unknown")
testing_dataset['Weather'] = testing_dataset['Weather'].fillna("Unknown")

In [ ]:
training_dataset['LargeVehicles'] = training_dataset['LargeVehicles'].replace({'Allowed': 1, 'Not Allowed': 0})
testing_dataset['LargeVehicles'] = testing_dataset['LargeVehicles'].replace({'Allowed': 1, 'Not Allowed': 0})

training_dataset['Landmarks'] = training_dataset['Landmarks'].replace({'Yes': 1, 'No': 0})
testing_dataset['Landmarks'] = testing_dataset['Landmarks'].replace({'Yes': 1, 'No': 0})

/tmp/ipykernel_15248/30838673.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  training_dataset['LargeVehicles'] = training_dataset['LargeVehicles'].replace({'Allowed': 1, 'Not Allowed': 0})
/tmp/ipykernel_15248/30838673.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  testing_dataset['LargeVehicles'] = testing_dataset['LargeVehicles'].replace({'Allowed': 1, 'Not Allowed': 0})
/tmp/ipykernel_15248/30838673.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain 

In [ ]:
training_dataset['Temperature'] = training_dataset.groupby('Weather')['Temperature'].transform(lambda x: x.fillna(x.mean()))
testing_dataset['Temperature'] = testing_dataset.groupby('Weather')['Temperature'].transform(lambda x: x.fillna(x.mean()))

display(training_dataset.head(5))

,Index,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location,hour,minute,part_of_day
0,0,48,0.048804,Unknown,1,0,0,16.750642,Unknown,2z1,0,0,Night
1,1,48,0.118507,Residential,3,1,1,31.104565,Sunny,2zt,0,0,Night
2,2,48,0.027132,Residential,1,0,0,25.919267,Sunny,8bj,0,0,Night
3,3,48,0.003272,Residential,1,0,0,10.934935,Rainy,8gt,0,0,Night
4,4,48,0.010819,Residential,1,0,0,10.803667,Rainy,2zq,0,0,Night


In [ ]:
training_dataset['active_hours'] = ((training_dataset['hour'] >= 8) & (training_dataset['hour'] <= 11)) | \
                                   ((training_dataset['hour'] >= 17) & (training_dataset['hour'] <= 20))

testing_dataset['active_hours'] = ((testing_dataset['hour'] >= 8) & (testing_dataset['hour'] <= 11)) | \
                                  ((testing_dataset['hour'] >= 17) & (testing_dataset['hour'] <= 20))

# Convert boolean to integer (1 or 0)
training_dataset['active_hours'] = training_dataset['active_hours'].astype(int)
testing_dataset['active_hours'] = testing_dataset['active_hours'].astype(int)

training_dataset.head(5)

,Index,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,geohash_location,hour,minute,part_of_day,active_hours
0,0,48,0.048804,Unknown,1,0,0,16.750642,Unknown,2z1,0,0,Night,0
1,1,48,0.118507,Residential,3,1,1,31.104565,Sunny,2zt,0,0,Night,0
2,2,48,0.027132,Residential,1,0,0,25.919267,Sunny,8bj,0,0,Night,0
3,3,48,0.003272,Residential,1,0,0,10.934935,Rainy,8gt,0,0,Night,0
4,4,48,0.010819,Residential,1,0,0,10.803667,Rainy,2zq,0,0,Night,0


In [ ]:
# Identify categorical columns to encode
categorical_cols = ['RoadType', 'Weather', 'part_of_day', 'geohash_location']

# One-hot encode the training and testing sets
training_dataset = pd.get_dummies(training_dataset, columns=categorical_cols)
testing_dataset = pd.get_dummies(testing_dataset, columns=categorical_cols)

# Ensure both dataframes have the same columns after encoding
# (Aligns test set to train set columns, filling missing with 0)
testing_dataset = testing_dataset.reindex(columns=[col for col in training_dataset.columns if col != 'demand'], fill_value=0)

training_dataset.head()

,Index,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,hour,minute,active_hours,...,geohash_location_djt,geohash_location_dju,geohash_location_djw,geohash_location_djy,geohash_location_dn0,geohash_location_dn4,geohash_location_dn5,geohash_location_dnh,geohash_location_dnj,geohash_location_dnn
0,0,48,0.048804,1,0,0,16.750642,0,0,0,...,False,False,False,False,False,False,False,False,False,False
1,1,48,0.118507,3,1,1,31.104565,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,2,48,0.027132,1,0,0,25.919267,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,3,48,0.003272,1,0,0,10.934935,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,4,48,0.010819,1,0,0,10.803667,0,0,0,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Calculate correlations with the 'demand' column
correlations = training_dataset.corr()['demand'].sort_values(ascending=False)

# Display the correlations
print("Correlation of features with demand:")
print(correlations)

Correlation of features with demand:
demand                  1.000000
RoadType_Highway        0.798625
RoadType_Street         0.290896
geohash_location_9d9    0.224823
NumberofLanes           0.214148
                          ...   
geohash_location_6pf   -0.018483
hour                   -0.037806
part_of_day_Night      -0.040817
part_of_day_Evening    -0.087790
RoadType_Residential   -0.756704
Name: demand, Length: 1272, dtype: float64


In [ ]:
# 1. Identify top 10 best and worst features (excluding 'demand' itself)
corr_series = correlations.drop('demand')
best_10 = corr_series.sort_values(ascending=False).head(10)
worst_10 = corr_series.sort_values(ascending=True).head(10)

# 2. Create 'best_features' (weighted sum of top 10 positive correlations)
training_dataset['best_features'] = training_dataset[best_10.index].mul(best_10).sum(axis=1)
testing_dataset['best_features'] = testing_dataset[best_10.index].mul(best_10).sum(axis=1)

# 3. Create 'worst_features' (weighted sum of top 10 negative correlations)
training_dataset['worst_features'] = training_dataset[worst_10.index].mul(worst_10).sum(axis=1)
testing_dataset['worst_features'] = testing_dataset[worst_10.index].mul(worst_10).sum(axis=1)

print("Top 10 Best Features applied:", list(best_10.index))
print("Top 10 Worst Features applied:", list(worst_10.index))
training_dataset[['best_features', 'worst_features', 'demand']].head()

Top 10 Best Features applied: ['RoadType_Highway', 'RoadType_Street', 'geohash_location_9d9', 'NumberofLanes', 'geohash_location_9ft', 'geohash_location_9e5', 'LargeVehicles', 'geohash_location_9d8', 'geohash_location_96x', 'geohash_location_9d2']
Top 10 Worst Features applied: ['RoadType_Residential', 'part_of_day_Evening', 'part_of_day_Night', 'hour', 'geohash_location_6pf', 'geohash_location_3m8', 'geohash_location_90n', 'geohash_location_3m2', 'geohash_location_3tx', 'geohash_location_3ms']


,best_features,worst_features,demand
0,0.214148,-0.040817,0.048804
1,0.836066,-0.797521,0.118507
2,0.214148,-0.797521,0.027132
3,0.214148,-0.797521,0.003272
4,0.214148,-0.797521,0.010819


In [ ]:
training_dataset.head(5)

,Index,day,demand,NumberofLanes,LargeVehicles,Landmarks,Temperature,hour,minute,active_hours,...,geohash_location_djw,geohash_location_djy,geohash_location_dn0,geohash_location_dn4,geohash_location_dn5,geohash_location_dnh,geohash_location_dnj,geohash_location_dnn,best_features,worst_features
0,0,48,0.048804,1,0,0,16.750642,0,0,0,...,False,False,False,False,False,False,False,False,0.214148,-0.040817
1,1,48,0.118507,3,1,1,31.104565,0,0,0,...,False,False,False,False,False,False,False,False,0.836066,-0.797521
2,2,48,0.027132,1,0,0,25.919267,0,0,0,...,False,False,False,False,False,False,False,False,0.214148,-0.797521
3,3,48,0.003272,1,0,0,10.934935,0,0,0,...,False,False,False,False,False,False,False,False,0.214148,-0.797521
4,4,48,0.010819,1,0,0,10.803667,0,0,0,...,False,False,False,False,False,False,False,False,0.214148,-0.797521


In [ ]:
# identifying the features and target for training and evaluation models
features = training_dataset.drop(['demand', 'Index'], axis=1)
target = training_dataset['demand']

test_features = testing_dataset.drop(['Index'], axis=1)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [ ]:
# Initialize the XGBoost Regressor model
xgb_model = xgb.XGBRegressor(colsample_bytree=1.0, n_estimators=800, max_depth=13, learning_rate=0.285, subsample=0.8, random_state=42)

# Train the model
xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.285, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=13,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# Make predictions on the test set
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate the model using Mean Squared Error
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
print(f"XGBoost Regressor Mean Squared Error: {mse_xgb}")

XGBoost Regressor Mean Squared Error: 0.0011092154140688473


In [ ]:
test_predictions = xgb_model.predict(test_features)

print("Predictions on the testing dataset have been made.")
print(f"First 5 predictions: {test_predictions[:5]}")

Predictions on the testing dataset have been made.
First 5 predictions: [0.03966705 0.01977994 0.04806798 0.02447964 0.04881941]


In [ ]:
submission = pd.DataFrame({
    'Index': testing_dataset['Index'],
    'demand': test_predictions
})

submission.to_csv('submission300504.csv', index=False)

print("Submission file 'submission300504.csv' created successfully.")
print(submission.head())

Submission file 'submission300504.csv' created successfully.
   Index    demand
0      0  0.039667
1      1  0.019780
2      2  0.048068
3      3  0.024480
4      4  0.048819
